In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets peft trl bitsandbytes accelerate

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Config
DATASET_PATH = "/content/drive/MyDrive/Lumen/merged_121.jsonl"
OUTPUT_DIR   = "/content/drive/MyDrive/Lumen/lumen-121-checkpoints"

BASE_MODEL   = "meta-llama/Meta-Llama-3.1-8B-Instruct"
HF_TOKEN     = ""  # paste your HuggingFace token here

In [ ]:
# 4. Load model in 4-bit (QLoRA)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 2048

print("dtype:", model.dtype)
print("VRAM allocated:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

In [ ]:
# 5. Load dataset
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

raw = load_jsonl(DATASET_PATH)
dataset = Dataset.from_list(raw)
dataset = dataset.train_test_split(test_size=0.01, seed=42)
print(f"Train: {len(dataset['train'])}  Val: {len(dataset['test'])}")

In [ ]:
# 6. Apply LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 7. Format dataset
def format_example(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )}

dataset = dataset.map(format_example)
print(dataset["train"][0]["text"][:500])

In [ ]:
# 8. Train — Lumen 1.2.1
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=1.5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=800,
    bf16=True,
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    load_best_model_at_end=True,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# 9. Save adapter
model.save_pretrained(OUTPUT_DIR + "/final")
tokenizer.save_pretrained(OUTPUT_DIR + "/final")
print("Saved to", OUTPUT_DIR + "/final")